In [23]:
from IPython.core.debugger import prompt
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

llm = init_chat_model(
    model = os.getenv("MODEL"),
    model_provider = "openai",
    api_key = os.getenv("API_KEY"),
    base_url = os.getenv("BASE_URL")
)

In [25]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda,RunnableBranch
prompt = ChatPromptTemplate.format_prompt("")
parser = StrOutputParser()
chain1 = llm | parser
chain2 = llm | parser

{'question': '什么是向量数据库', 'context': '[关于「什么是向量数据库」检索到的资料……]'}


In [ ]:
# ============================================================
# RunnableBranch：按内容分叉走不同链
# 场景：邮件分类 —— 求职/合作/其他，各走各的回复链
# ============================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch

parser = StrOutputParser()

# 1. 三条不同用途的回复链
job_chain = (
    ChatPromptTemplate.from_template(
        "你是一个HR，收到一封求职邮件：「{email}」请礼貌回复，说明已收到简历，一周内答复。"
    )
    | llm
    | parser
)

biz_chain = (
    ChatPromptTemplate.from_template(
        "你是一个商务经理，收到一封商务合作邮件：「{email}」请回复表示感兴趣，并安排电话沟通时间。"
    )
    | llm
    | parser
)

fallback_chain = (
    ChatPromptTemplate.from_template(
        "你是一个行政助理，收到一封普通邮件：「{email}」请简短友好地回复。"
    )
    | llm
    | parser
)

# 2. 条件判断（输入是 {"email": "..."}）
def is_job_application(x: dict) -> bool:
    """包含求职关键词 → 简历处理链"""
    keywords = ["求职", "应聘", "简历", "面试", "岗位"]
    return any(kw in x["email"] for kw in keywords)


def is_business(x: dict) -> bool:
    """包含商务关键词 → 合作洽谈链"""
    keywords = ["合作", "采购", "报价", "商务", "方案", "预算"]
    return any(kw in x["email"] for kw in keywords)


# 3. 组装分支：按顺序匹配，命中即走对应链
branch = RunnableBranch(
    (is_job_application, job_chain),   # 优先级1：求职
    (is_business, biz_chain),          # 优先级2：商务
    fallback_chain,                    # 默认：普通邮件
)

# 4. 测试
emails = [
    "你好，我对贵公司的Java开发岗位很感兴趣，这是我的简历",
    "我们公司想采购一批办公设备，能否提供报价？",
    "周末团建去哪里，大家投个票",
]

for email in emails:
    print(f"\n📧 原始邮件：{email}")
    print(f"📬 自动回复：{branch.invoke({'email': email})}")
    print("-" * 50)